# ReefScan-ID: Direct 5-class SegFormer-B0 (E4 a)

SegFormer-B0 trained to label every pixel as coral healthy, coral bleached, coral dead, substrate or other, for the direct side of E4. The data, crops, loss and scoring follow notebook 02, with the 5-class mapping of the 2026-11-19 decision. The test split is not used. Run in Google Colab with a GPU runtime.

### Setup: Colab runtime and Google Drive

In [1]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))
print("CPU count: {}".format(os.cpu_count()))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))

Mounted at /content/drive
GPU: NVIDIA L4
CPU count: 12
Drive folder found: True


### Setup: libraries

In [2]:
!pip install -q datasets
import random
import time
import numpy as np
from PIL import Image

import datasets
print("torch version: {}".format(torch.__version__))
print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))

torch version: 2.14.1+cu130
datasets version: 4.8.5
numpy version: 2.4.6


### Settings

The optimiser settings are copied from notebook 02: AdamW, learning rate 6e-5, weight decay 0.01, linear decay to 0. The number of epochs is set after the timing cell.

In [3]:
crop_size = 512
batch_size = 8
num_workers = 2
val_height = 512
val_width = 1024
val_batch_size = 4
seed = 0

learning_rate = 6e-5
weight_decay = 0.01
timing_steps = 30

# Reference values printed by notebook 02
train_pixels_reference = 3181379584
segformer3_params = 3714915

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("Crops: {} by {}, batch size {}, workers {}".format(crop_size, crop_size, batch_size, num_workers))
print("Validation photos resized to {} by {}, batch size {}".format(val_height, val_width, val_batch_size))
print("Optimiser: AdamW, learning rate {}, weight decay {}, linear decay to 0 (as in notebook 02)".format(
    learning_rate, weight_decay))
print("The number of epochs is not set here")
print("The test split is not used in this notebook")

Crops: 512 by 512, batch size 8, workers 2
Validation photos resized to 512 by 1024, batch size 4
Optimiser: AdamW, learning rate 6e-05, weight decay 0.01, linear decay to 0 (as in notebook 02)
The number of epochs is not set here
The test split is not used in this notebook


## Section 1: Data, 5-class mapping, model and timing

Question: does the direct model see the same data, crops, loss and scoring as the 3-class SegFormer-B0 of notebook 02 with the 5-class mapping of the 2026-11-19 decision, how are the train pixels spread over the five classes, and how long does one epoch take on this GPU?

Checks: load Coralscapes from the Drive cache, build the 5-class mapping and compare it with notebook 02's coral group, count the train pixels and compute the class weights with the formula of notebook 02, build the crop and validation loaders, build SegFormer-B0 with five classes and run one forward pass, score the untrained model once, then time 30 training steps and one validation pass to estimate the seconds per epoch. Only the train and validation splits are used.

### Load the dataset

In [4]:
raw_dir = base_dir + "/RawData"

from datasets import load_dataset

print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

Cache folder exists: True


README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})
Loading took 30.5 seconds


### Class names from the dataset card

Ids and names copied from notebook 02, which copied them from the table on the Coralscapes dataset card.

In [5]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Class groups

The coral ids are split into healthy, bleached and dead with the stage 2 state map of notebook 04. Substrate and the coral group come from the stage 1 map of notebook 02. The excluded coral ids come from `settings.json`.

In [6]:
import json

# Stage 2 state map of notebook 04
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]

# Stage 1 groups of notebook 02
substrate_ids = [5, 10, 12, 18]
coral_ids = [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]

with open(base_dir + "/Processed/stage2_patches/settings.json") as handle:
    settings = json.load(handle)
excluded_ids = settings["excluded_ids"]

print("Healthy ids: {}".format(healthy_ids))
print("Bleached ids: {}".format(bleached_ids))
print("Dead ids: {}".format(dead_ids))
print("Substrate ids: {}".format(substrate_ids))
print("Coral ids of notebook 02: {}".format(coral_ids))
print("Excluded ids from settings.json: {}".format(excluded_ids))

Healthy ids: [6, 17, 22, 25, 28, 31, 34, 36]
Bleached ids: [4, 16, 19, 33]
Dead ids: [3, 20, 23, 32, 37]
Substrate ids: [5, 10, 12, 18]
Coral ids of notebook 02: [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]
Excluded ids from settings.json: [21, 27]


### Build the 5-class mapping

In [7]:
group_names = {0: "coral healthy", 1: "coral bleached", 2: "coral dead", 3: "substrate", 4: "other"}
ignore_id = 255

# Everything on the card starts as other. Value 0 and the excluded coral ids are ignored
mapping = np.full(256, ignore_id, dtype=np.uint8)
for value in class_names:
    mapping[value] = 4
mapping[substrate_ids] = 3
mapping[healthy_ids] = 0
mapping[bleached_ids] = 1
mapping[dead_ids] = 2
mapping[excluded_ids] = ignore_id
mapping[0] = ignore_id

for group_id, group_name in group_names.items():
    names = [class_names[v] for v in class_names if mapping[v] == group_id]
    print("{} {} ({} ids): {}".format(group_id, group_name, len(names), names))
ignored = [class_names[v] for v in class_names if mapping[v] == ignore_id]
print("ignored ({} ids on the card): {}".format(len(ignored), ignored))
print("Mask value 0 maps to: {}".format(mapping[0]))

coral_five = set(healthy_ids + bleached_ids + dead_ids)
print("No id is in two classes: {}".format(
    len(healthy_ids + bleached_ids + dead_ids + substrate_ids) == len(coral_five | set(substrate_ids))))
print("Coral classes plus the ignored coral ids equal notebook 02's coral group: {}".format(
    coral_five | set(excluded_ids) == set(coral_ids)))
print("The excluded ids of settings.json are 21 and 27, as in the decision: {}".format(sorted(excluded_ids) == [21, 27]))
print("Class 4 holds the ids that are neither coral nor substrate in notebook 02: {}".format(
    {v for v in class_names if mapping[v] == 4} == set(class_names) - set(coral_ids) - set(substrate_ids)))

0 coral healthy (8 ids): ['other coral alive', 'massive/meandering alive', 'branching alive', 'acropora alive', 'table acropora alive', 'pocillopora alive', 'stylophora alive', 'meandering alive']
1 coral bleached (4 ids): ['other coral bleached', 'massive/meandering bleached', 'branching bleached', 'meandering bleached']
2 coral dead (5 ids): ['other coral dead', 'branching dead', 'massive/meandering dead', 'table acropora dead', 'meandering dead']
3 substrate (4 ids): ['sand', 'algae covered substrate', 'unknown hard substrate', 'rubble']
4 other (16 ids): ['seagrass', 'trash', 'human', 'transect tools', 'fish', 'other animal', 'background', 'dark', 'transect line', 'clam', 'sea cucumber', 'sponge', 'anemone', 'sea urchin', 'crown of thorn', 'dead clam']
ignored (2 ids on the card): ['millepora', 'turbinaria']
Mask value 0 maps to: 255
No id is in two classes: True
Coral classes plus the ignored coral ids equal notebook 02's coral group: True
The excluded ids of settings.json are 21 

### Count train pixels per mask value

In [8]:
pixel_counts = {}

start = time.time()
counts = np.zeros(256, dtype=np.int64)
masks = ds["train"].select_columns(["label"])
for row in masks:
    mask = np.array(row["label"])
    counts += np.bincount(mask.ravel(), minlength=256)
pixel_counts["train"] = counts

print("train: {} masks counted, {} pixels, {:.0f} seconds".format(masks.num_rows, counts.sum(), time.time() - start))
print("Same pixel total as notebook 02: {}".format(int(counts.sum()) == train_pixels_reference))

train: 1517 masks counted, 3181379584 pixels, 44 seconds
Same pixel total as notebook 02: True


### Pixel share per class

In [9]:
for split in ["train"]:
    counts = pixel_counts[split]
    total = counts.sum()
    for group_id, group_name in group_names.items():
        share = 100 * counts[mapping == group_id].sum() / total
        print("{} {}: {:.2f}% of all pixels".format(split, group_name, share))
    ignored = 100 * counts[mapping == ignore_id].sum() / total
    print("{} ignored: {:.2f}% of all pixels".format(split, ignored))

train coral healthy: 21.19% of all pixels
train coral bleached: 0.83% of all pixels
train coral dead: 3.16% of all pixels
train substrate: 27.53% of all pixels
train other: 29.81% of all pixels
train ignored: 17.48% of all pixels


### Class weights and loss

In [10]:
train_counts = pixel_counts["train"]
group_pixels = np.array([train_counts[mapping == group_id].sum() for group_id in group_names])
group_share = group_pixels / group_pixels.sum()
class_weights = 1 / group_share
class_weights = class_weights / class_weights.mean()

for group_id, group_name in group_names.items():
    print("{}: {:.2f}% of labelled train pixels, weight {:.3f}".format(
        group_name, 100 * group_share[group_id], class_weights[group_id]))

weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)


def segmentation_loss(logits, masks):
    upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
    return torch.nn.functional.cross_entropy(upsampled, masks, weight=weight_tensor, ignore_index=ignore_id)

coral healthy: 25.68% of labelled train pixels, weight 0.144
coral bleached: 1.00% of labelled train pixels, weight 3.681
coral dead: 3.83% of labelled train pixels, weight 0.963
substrate: 33.37% of labelled train pixels, weight 0.110
other: 36.13% of labelled train pixels, weight 0.102


### Crop dataset class

In [11]:
import random
from torch.utils.data import Dataset

crop_size = 512


class CropDataset(Dataset):
    def __init__(self, split, crop_size, flip):
        self.split = split
        self.crop_size = crop_size
        self.flip = flip

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = np.array(row["image"])
        mask = mapping[np.array(row["label"])]

        # Same random window for the image and the mask
        top = random.randint(0, image.shape[0] - self.crop_size)
        left = random.randint(0, image.shape[1] - self.crop_size)
        image = image[top:top + self.crop_size, left:left + self.crop_size]
        mask = mask[top:top + self.crop_size, left:left + self.crop_size]

        if self.flip and random.random() < 0.5:
            image = image[:, ::-1]
            mask = mask[:, ::-1]

        image = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mask.copy()).long()
        return image, mask


print("Crop size: {}".format(crop_size))

Crop size: 512


### Validation dataset class

In [12]:
class ResizeDataset(Dataset):
    def __init__(self, split, height, width):
        self.split = split
        self.height = height
        self.width = width

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = row["image"].resize((self.width, self.height), Image.BILINEAR)
        label = row["label"].resize((self.width, self.height), Image.NEAREST)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mapping[np.array(label)]).long()
        return image, mask


print("Validation dataset class defined")

Validation dataset class defined


### Data loaders

In [13]:
from torch.utils.data import DataLoader

train_data = CropDataset(ds["train"], crop_size, flip=True)
eval_data = ResizeDataset(ds["validation"], val_height, val_width)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
eval_loader = DataLoader(eval_data, batch_size=val_batch_size, shuffle=False, num_workers=num_workers)

print("Train crops per epoch: {}, batches: {}".format(len(train_data), len(train_loader)))
print("Validation photos: {}, batches: {}".format(len(eval_data), len(eval_loader)))

for name, loader in [("train", train_loader), ("validation", eval_loader)]:
    images, masks = next(iter(loader))
    print("{} batch: images {}, dtype {}, min {:.3f}, max {:.3f}".format(
        name, tuple(images.shape), images.dtype, images.min().item(), images.max().item()))
    print("{} batch: masks {}, dtype {}, values {}".format(
        name, tuple(masks.shape), masks.dtype, torch.unique(masks).tolist()))

Train crops per epoch: 1517, batches: 190
Validation photos: 166, batches: 42
train batch: images (8, 3, 512, 512), dtype torch.float32, min 0.000, max 1.000
train batch: masks (8, 512, 512), dtype torch.int64, values [0, 1, 2, 3, 4, 255]
validation batch: images (4, 3, 512, 1024), dtype torch.float32, min 0.000, max 1.000
validation batch: masks (4, 512, 1024), dtype torch.int64, values [0, 1, 2, 3, 4, 255]


### SegFormer-B0 with five classes

In [14]:
import transformers
from transformers import SegformerForSemanticSegmentation

weights_name = "nvidia/mit-b0"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    weights_name,
    num_labels=len(group_names),
    id2label=group_names,
    label2id={name: group_id for group_id, name in group_names.items()},
    ignore_mismatched_sizes=True,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("transformers version: {}".format(transformers.__version__))
print("Weights: {}".format(weights_name))
print("Device: {}".format(device))
print("Parameters: {} total, {} trainable".format(total_params, trainable_params))
print("Parameters of the 3-class SegFormer-B0 printed by notebook 02: {}".format(segformer3_params))
print("Difference to the 5-class model: {}".format(total_params - segformer3_params))

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.linear_fuse.weight                          

transformers version: 5.22.1
Weights: nvidia/mit-b0
Device: cuda
Parameters: 3715429 total, 3715429 trainable
Parameters of the 3-class SegFormer-B0 printed by notebook 02: 3714915
Difference to the 5-class model: 514


### One forward pass

In [15]:
image_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
image_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

images, masks = next(iter(train_loader))
pixel_values = (images.to(device) - image_mean) / image_std

model.eval()
with torch.no_grad():
    logits = model(pixel_values=pixel_values).logits
upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

print("Input shape: {}".format(tuple(pixel_values.shape)))
print("Logits shape: {}".format(tuple(logits.shape)))
print("Upsampled logits shape: {}".format(tuple(upsampled.shape)))
print("Logits have one channel per class: {}".format(logits.shape[1] == len(group_names)))
print("Upsampled logits have the size of the masks: {}".format(tuple(upsampled.shape[-2:]) == tuple(masks.shape[-2:])))

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Input shape: (8, 3, 512, 512)
Logits shape: (8, 5, 128, 128)
Upsampled logits shape: (8, 5, 512, 512)
Logits have one channel per class: True
Upsampled logits have the size of the masks: True


### Validation metrics

In [16]:
group_count = len(group_names)


def evaluate(model, loader):
    confusion = torch.zeros(group_count, group_count, dtype=torch.int64, device=device)
    model.eval()
    with torch.no_grad():
        for images, masks in loader:
            pixel_values = (images.to(device) - image_mean) / image_std
            masks = masks.to(device)
            logits = model(pixel_values=pixel_values).logits
            upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled.argmax(dim=1)

            # Pixels with the ignore id are left out
            keep = masks != ignore_id
            pairs = masks[keep] * group_count + predicted[keep]
            confusion += torch.bincount(pairs, minlength=group_count ** 2).reshape(group_count, group_count)

    intersection = confusion.diag().float()
    union = confusion.sum(dim=0).float() + confusion.sum(dim=1).float() - intersection
    return (intersection / union).cpu().numpy()


start = time.time()
iou = evaluate(model, eval_loader)
print("Model before training")
for group_id, group_name in group_names.items():
    print("{} IoU: {:.4f}".format(group_name, iou[group_id]))
print("mIoU: {:.4f}".format(iou.mean()))
print("One validation pass took {:.1f} seconds".format(time.time() - start))

Model before training
coral healthy IoU: 0.0191
coral bleached IoU: 0.0177
coral dead IoU: 0.0109
substrate IoU: 0.2075
other IoU: 0.0119
mIoU: 0.0534
One validation pass took 13.9 seconds


### Step timing and estimated seconds per epoch

The training steps run on a copy of the model, so the model above stays untouched. The data loading time is the time the training loop waits for the next batch, so it already includes the overlap with the model step.

In [17]:
import copy

timing_model = copy.deepcopy(model)
timing_model.train()
timing_optimizer = torch.optim.AdamW(timing_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

data_seconds = []
step_seconds = []
batches = iter(train_loader)
for _ in range(timing_steps):
    start = time.time()
    images, masks = next(batches)
    data_seconds.append(time.time() - start)

    start = time.time()
    pixel_values = (images.to(device) - image_mean) / image_std
    loss = segmentation_loss(timing_model(pixel_values=pixel_values).logits, masks.to(device))
    timing_optimizer.zero_grad()
    loss.backward()
    timing_optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    step_seconds.append(time.time() - start)

start = time.time()
evaluate(timing_model, eval_loader)
validation_seconds = time.time() - start

first_step = data_seconds[0] + step_seconds[0]
steady_step = np.mean(data_seconds[1:]) + np.mean(step_seconds[1:])
steps_per_epoch = len(train_loader)
epoch_seconds = first_step + (steps_per_epoch - 1) * steady_step + validation_seconds

print("Steps timed: {}, batch size: {}".format(timing_steps, batch_size))
print("Step 1, includes start-up: data loading {:.2f} seconds, model step {:.2f} seconds".format(
    data_seconds[0], step_seconds[0]))
print("Steps 2 to {}: data loading {:.2f} seconds per step".format(timing_steps, np.mean(data_seconds[1:])))
print("Steps 2 to {}: model step {:.2f} seconds per step".format(timing_steps, np.mean(step_seconds[1:])))
print("Steps 2 to {}: total {:.2f} seconds per training step".format(timing_steps, steady_step))
print("One validation pass: {:.1f} seconds".format(validation_seconds))
if device == "cuda":
    print("Peak GPU memory during the training steps: {:.1f} GB".format(torch.cuda.max_memory_allocated() / 1024 ** 3))
print("Estimated seconds per epoch: {:.0f} ({} steps and one validation pass)".format(epoch_seconds, steps_per_epoch))
print("Estimated training time for a number of epochs on {}:".format(gpu_name))
for epochs_option in [10, 20, 30, 40, 50]:
    print("  {} epochs: about {:.0f} minutes".format(epochs_option, epochs_option * epoch_seconds / 60))

Steps timed: 30, batch size: 8
Step 1, includes start-up: data loading 0.85 seconds, model step 0.87 seconds
Steps 2 to 30: data loading 0.41 seconds per step
Steps 2 to 30: model step 0.13 seconds per step
Steps 2 to 30: total 0.54 seconds per training step
One validation pass: 10.0 seconds
Peak GPU memory during the training steps: 2.8 GB
Estimated seconds per epoch: 114 (190 steps and one validation pass)
Estimated training time for a number of epochs on NVIDIA L4:
  10 epochs: about 19 minutes
  20 epochs: about 38 minutes
  30 epochs: about 57 minutes
  40 epochs: about 76 minutes
  50 epochs: about 95 minutes


**Observation**: We mapped Coralscapes into five classes and ignored Millepora, Turbinaria, and unlabeled pixels. Bleached coral made up only 0.83% of training pixels, so we used class-weighted loss. The 5-class SegFormer-B0 passed the input and output checks, but its mIoU before training was only 0.0534. On the L4, one epoch is estimated to take 114 seconds. We still need to train the model before comparing it with the cascade.

## Section 2: Training

Question: how do the training loss and the validation mIoU and per-class IoU of the direct 5-class SegFormer-B0 change over 50 epochs, and which epoch is best?

Checks: set the number of epochs and rebuild the loaders with more workers, time 20 training steps again with the new loaders, build the optimiser and the linear decay schedule, set up the checkpoints with resume, train with a checkpoint after every epoch, then plot the train loss and the validation IoU. The datasets, the batch sizes, the loss and the scoring are the ones of Section 1, so run the cells of Section 1 first. Only the train and validation splits are used.

### Training settings

The loaders are rebuilt with 8 workers instead of 2 because the data loading was most of each step in Section 1. The datasets, the batch sizes, the shuffling and the seed are the same.

In [ ]:
epochs = 50
resume = True
num_workers = 8
timing_steps = 20

# Printed by Section 1, which used 2 workers
section1_workers = 2
section1_epoch_seconds = 114

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
eval_loader = DataLoader(eval_data, batch_size=val_batch_size, shuffle=False, num_workers=num_workers)

print("Epochs: {}, resume from the last checkpoint: {}".format(epochs, resume))
print("Workers: {} (Section 1: {}), CPU count: {}".format(num_workers, section1_workers, os.cpu_count()))
print("Batch size: train {}, validation {}. Shuffling: train yes, validation no. Seed: {}".format(
    batch_size, val_batch_size, seed))
print("Same datasets as Section 1: {} train crops, {} validation photos".format(len(train_data), len(eval_data)))
print("Train batches per epoch: {}, validation batches: {}".format(len(train_loader), len(eval_loader)))

### Step timing with the new loaders

The training steps run on a copy of the model. The estimate adds the first step, the other steps and one validation pass, as in Section 1.

In [ ]:
import copy

timing_model = copy.deepcopy(model)
timing_model.train()
timing_optimizer = torch.optim.AdamW(timing_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

data_seconds = []
step_seconds = []
batches = iter(train_loader)
for _ in range(timing_steps):
    start = time.time()
    images, masks = next(batches)
    data_seconds.append(time.time() - start)

    start = time.time()
    pixel_values = (images.to(device) - image_mean) / image_std
    loss = segmentation_loss(timing_model(pixel_values=pixel_values).logits, masks.to(device))
    timing_optimizer.zero_grad()
    loss.backward()
    timing_optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    step_seconds.append(time.time() - start)

start = time.time()
evaluate(timing_model, eval_loader)
validation_seconds = time.time() - start

first_step = data_seconds[0] + step_seconds[0]
steady_step = np.mean(data_seconds[1:]) + np.mean(step_seconds[1:])
steps_per_epoch = len(train_loader)
epoch_seconds = first_step + (steps_per_epoch - 1) * steady_step + validation_seconds

print("Steps timed: {}, batch size: {}".format(timing_steps, batch_size))
print("Step 1, includes start-up: data loading {:.2f} seconds, model step {:.2f} seconds".format(
    data_seconds[0], step_seconds[0]))
print("Steps 2 to {}: data loading {:.2f} seconds per step".format(timing_steps, np.mean(data_seconds[1:])))
print("Steps 2 to {}: model step {:.2f} seconds per step".format(timing_steps, np.mean(step_seconds[1:])))
print("Steps 2 to {}: total {:.2f} seconds per training step".format(timing_steps, steady_step))
print("One validation pass: {:.1f} seconds".format(validation_seconds))
if device == "cuda":
    print("Peak GPU memory during the training steps: {:.1f} GB".format(torch.cuda.max_memory_allocated() / 1024 ** 3))
print("Estimated seconds per epoch: {:.0f} ({} steps and one validation pass)".format(epoch_seconds, steps_per_epoch))
print("Estimated training time for a number of epochs on {}:".format(gpu_name))
for epochs_option in [10, 20, 30, 40, 50]:
    print("  {} epochs: about {:.0f} minutes".format(epochs_option, epochs_option * epoch_seconds / 60))
print("Section 1 estimate with {} workers: {} seconds per epoch, {:.0f} minutes for {} epochs".format(
    section1_workers, section1_epoch_seconds, epochs * section1_epoch_seconds / 60, epochs))
print("New estimate with {} workers: {:.0f} seconds per epoch, {:.0f} minutes for {} epochs".format(
    num_workers, epoch_seconds, epochs * epoch_seconds / 60, epochs))

### Optimiser and schedule

In [ ]:
total_steps = epochs * len(train_loader)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: max(0.0, 1 - step / total_steps))

print("Epochs: {}, steps per epoch: {}, total steps: {}".format(epochs, len(train_loader), total_steps))
print("Optimiser: AdamW, learning rate {}, weight decay {}, linear decay to 0".format(learning_rate, weight_decay))

### Checkpoint and resume

In [ ]:
checkpoint_dir = base_dir + "/Checkpoints/stage1_direct5"
os.makedirs(checkpoint_dir, exist_ok=True)
last_path = checkpoint_dir + "/last.pt"
best_path = checkpoint_dir + "/best.pt"


def save_checkpoint(state, file_path):
    # A temporary file is written first, so that a disconnect cannot leave a broken checkpoint
    torch.save(state, file_path + ".tmp")
    os.replace(file_path + ".tmp", file_path)


start_epoch = 0
global_step = 0
best_miou = 0.0
history = []

if resume and os.path.isfile(last_path):
    checkpoint = torch.load(last_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    start_epoch = checkpoint["epoch"]
    global_step = checkpoint["global_step"]
    best_miou = checkpoint["best_miou"]
    history = checkpoint["history"]
    print("Resumed after epoch {}, step {}, best mIoU {:.4f}".format(start_epoch, global_step, best_miou))
else:
    print("No checkpoint loaded: training starts from the pretrained encoder and a new head")

print("Checkpoint folder: {}".format(checkpoint_dir))

### Training loop

In [ ]:
for epoch in range(start_epoch, epochs):
    epoch_start = time.time()
    model.train()
    loss_sum = 0.0
    for images, masks in train_loader:
        pixel_values = (images.to(device) - image_mean) / image_std
        logits = model(pixel_values=pixel_values).logits
        loss = segmentation_loss(logits, masks.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        global_step += 1
        loss_sum += loss.item()

    iou = evaluate(model, eval_loader)
    miou = float(iou.mean())
    history.append({
        "epoch": epoch + 1,
        "train_loss": loss_sum / len(train_loader),
        "miou": miou,
        "ious": [float(value) for value in iou],
        "seconds": time.time() - epoch_start,
    })

    is_best = miou > best_miou
    if is_best:
        best_miou = miou

    state = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch + 1,
        "global_step": global_step,
        "best_miou": best_miou,
        "history": history,
    }
    save_checkpoint(state, last_path)
    if is_best:
        save_checkpoint(state, best_path)

    record = history[-1]
    class_text = ", ".join("{} {:.4f}".format(name, value) for name, value in zip(group_names.values(), record["ious"]))
    print("Epoch {}: loss {:.4f}, mIoU {:.4f}, {}, {:.0f} seconds, best saved: {}".format(
        record["epoch"], record["train_loss"], record["miou"], class_text, record["seconds"], is_best))

print("Epochs done: {}, best validation mIoU: {:.4f}".format(len(history), best_miou))

### Training curves

In [ ]:
import matplotlib.pyplot as plt

epoch_numbers = [record["epoch"] for record in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epoch_numbers, [record["train_loss"] for record in history])
axes[0].set_title("Train loss, weighted cross-entropy")
axes[0].set_xlabel("Epoch")
axes[1].plot(epoch_numbers, [record["miou"] for record in history], label="mIoU", color="black", linewidth=2)
for class_id, class_name in group_names.items():
    axes[1].plot(epoch_numbers, [record["ious"][class_id] for record in history], label=class_name)
axes[1].set_title("Validation IoU, photos resized to {} by {}".format(val_height, val_width))
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

best_record = max(history, key=lambda record: record["miou"])
print("Best epoch: {}, mIoU {:.4f}".format(best_record["epoch"], best_record["miou"]))
for class_id, class_name in group_names.items():
    print("  {} IoU: {:.4f}".format(class_name, best_record["ious"][class_id]))

**Observation**:

## Section 3: Best epoch and validation table

Question: which epoch of the direct 5-class model is best on the validation photos, what are its IoU per class, and how does it compare with the 3-class SegFormer-B0 of stage 1 when its prediction is merged into coral, substrate and other?

Checks: read the checkpoints and check that best.pt holds the best epoch of the history and that the parameter count matches Section 1, print the 5-class table, run one validation pass of best.pt against the 3-class masks of notebook 02 after merging coral healthy, bleached and dead into coral, read the 3-class SegFormer-B0 checkpoint without changing it, and print the comparison table. Only the validation split is used and the test split is not touched. Run the cells of Section 1 first: the validation pass needs its data, loaders and model.

### Checkpoint files

In [ ]:
import datetime

direct5_dir = base_dir + "/Checkpoints/stage1_direct5"
print("This section only reads checkpoints; it writes nothing")

for kind in ["best", "last"]:
    file_path = "{}/{}.pt".format(direct5_dir, kind)
    if os.path.isfile(file_path):
        stamp = datetime.datetime.fromtimestamp(os.path.getmtime(file_path)).strftime("%Y-%m-%d %H:%M")
        print("5-class {}.pt: found, {:.1f} MB, modified {}".format(kind, os.path.getsize(file_path) / 1024 ** 2, stamp))
    else:
        print("5-class {}.pt: not found".format(kind))

### 5-class model: best epoch

In [ ]:
# Parameters printed by Section 1
direct5_params_section1 = 3715429
buffer_names = ("running_mean", "running_var", "num_batches_tracked")

direct5_best_state = torch.load(direct5_dir + "/best.pt", map_location="cpu", weights_only=False)
direct5_last_state = torch.load(direct5_dir + "/last.pt", map_location="cpu", weights_only=False)

direct5_history = direct5_last_state["history"]
direct5_best = max(direct5_history, key=lambda record: record["miou"])
direct5_epochs = direct5_last_state["epoch"]
direct5_params = sum(tensor.numel() for key, tensor in direct5_last_state["model"].items() if not key.endswith(buffer_names))

print("last.pt holds epoch {} and a history of {} entries".format(direct5_epochs, len(direct5_history)))
print("Best epoch by validation mIoU: {} (mIoU {:.4f})".format(direct5_best["epoch"], direct5_best["miou"]))
print("best.pt holds epoch {}, the best epoch: {}".format(direct5_best_state["epoch"], direct5_best_state["epoch"] == direct5_best["epoch"]))
print("Parameters counted in the checkpoint: {}, same as Section 1 ({}): {}".format(
    direct5_params, direct5_params_section1, direct5_params == direct5_params_section1))

### Validation table: 5 classes

In [ ]:
import pandas as pd

row = {
    "model": "SegFormer-B0 direct, 5 classes",
    "parameters": direct5_params,
    "epochs trained": direct5_epochs,
    "best epoch": direct5_best["epoch"],
    "mIoU": direct5_best["miou"],
}
for class_id, class_name in group_names.items():
    row[class_name + " IoU"] = direct5_best["ious"][class_id]

print("5-class model at its best epoch on the {} validation photos, resized to {} by {}".format(
    len(eval_data), val_height, val_width))
print("The test split is not used")
print(pd.DataFrame([row]).round(4).to_string(index=False))

### Validation pass with the 3-class masks

The 3-class masks are the ones of notebook 02: Millepora and Turbinaria count as coral and mask value 0 is ignored. The prediction of the 5-class model is merged: classes 0, 1 and 2 become coral, class 3 substrate and class 4 other.

In [ ]:
import copy

group_names3 = {0: "coral", 1: "substrate", 2: "other"}

# Same rules as notebook 02: coral ids include Millepora and Turbinaria, value 0 is ignored
mapping3 = np.full(256, ignore_id, dtype=np.uint8)
for value in class_names:
    mapping3[value] = 2
mapping3[coral_ids] = 0
mapping3[substrate_ids] = 1


class ResizeDataset3(Dataset):
    def __init__(self, split, height, width):
        self.split = split
        self.height = height
        self.width = width

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = row["image"].resize((self.width, self.height), Image.BILINEAR)
        label = row["label"].resize((self.width, self.height), Image.NEAREST)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mapping3[np.array(label)]).long()
        return image, mask


eval_data3 = ResizeDataset3(ds["validation"], val_height, val_width)
eval_loader3 = DataLoader(eval_data3, batch_size=val_batch_size, shuffle=False, num_workers=num_workers)
print("Validation photos: {}, batches: {}".format(len(eval_data3), len(eval_loader3)))
print("Millepora (21) and Turbinaria (27) are coral in the 3-class masks: {}".format(
    mapping3[21] == 0 and mapping3[27] == 0))
print("Mask value 0 maps to: {}".format(mapping3[0]))

best_model = copy.deepcopy(model)
best_model.load_state_dict(direct5_best_state["model"])
best_model = best_model.to(device)
print("Weights loaded from best.pt, epoch {}".format(direct5_best_state["epoch"]))
del direct5_best_state, direct5_last_state

merge_table = torch.tensor([0, 0, 0, 1, 2], device=device)


def evaluate_merged(model, loader):
    confusion = torch.zeros(3, 3, dtype=torch.int64, device=device)
    model.eval()
    with torch.no_grad():
        for images, masks in loader:
            masks = masks.to(device)
            pixel_values = (images.to(device) - image_mean) / image_std
            logits = model(pixel_values=pixel_values).logits
            upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            predicted = merge_table[upsampled.argmax(dim=1)]

            # Pixels with the ignore id are left out
            keep = masks != ignore_id
            pairs = masks[keep] * 3 + predicted[keep]
            confusion += torch.bincount(pairs, minlength=9).reshape(3, 3)

    intersection = confusion.diag().float()
    union = confusion.sum(dim=0).float() + confusion.sum(dim=1).float() - intersection
    return (intersection / union).cpu().numpy(), confusion.cpu().numpy()


start = time.time()
merged_iou, merged_confusion = evaluate_merged(best_model, eval_loader3)
print("One validation pass of best.pt took {:.1f} seconds".format(time.time() - start))
for class_id, class_name in group_names3.items():
    print("{} IoU: {:.4f}".format(class_name, merged_iou[class_id]))
print("mIoU: {:.4f}".format(merged_iou.mean()))
print("Pixels scored: {}".format(int(merged_confusion.sum())))

### SegFormer-B0 with 3 classes: best epoch, read only

In [ ]:
# Values printed by notebook 02 section 5 and recorded in the handoff
reference = {"epoch": 37, "miou": 0.6775, "coral_iou": 0.6284, "substrate_iou": 0.6733, "other_iou": 0.7309}

segformer3_dir = base_dir + "/Checkpoints/stage1_segformer_b0"
segformer3_best_state = torch.load(segformer3_dir + "/best.pt", map_location="cpu", weights_only=False)
segformer3_last_state = torch.load(segformer3_dir + "/last.pt", map_location="cpu", weights_only=False)
segformer3_epochs = segformer3_last_state["epoch"]
print("last.pt holds epoch {}, best.pt holds epoch {}".format(segformer3_epochs, segformer3_best_state["epoch"]))

if len(segformer3_last_state.get("history", [])) > 0:
    segformer3_history = segformer3_last_state["history"]
    segformer3_best = max(segformer3_history, key=lambda record: record["miou"])
    segformer3_source = "the checkpoint history"
    print("History: {} entries, epochs {} to {}".format(
        len(segformer3_history), segformer3_history[0]["epoch"], segformer3_history[-1]["epoch"]))
    print("Best epoch by validation mIoU: {} (mIoU {:.4f})".format(segformer3_best["epoch"], segformer3_best["miou"]))
    print("best.pt holds the best epoch: {}".format(segformer3_best_state["epoch"] == segformer3_best["epoch"]))
    found = [segformer3_best[key] for key in reference]
    print("Same as notebook 02 section 5 (epoch 37, mIoU 0.6775, coral 0.6284, substrate 0.6733, other 0.7309): {}".format(
        found[0] == reference["epoch"] and [round(value, 4) for value in found[1:]] == list(reference.values())[1:]))
else:
    print("The SegFormer-B0 checkpoint holds no history, so the values printed by notebook 02 section 5 are used")
    segformer3_best = dict(reference)
    segformer3_source = "the values printed by notebook 02 section 5"
del segformer3_best_state, segformer3_last_state

### Validation table: merged to 3 classes

In [ ]:
rows = []
runs = [
    ("SegFormer-B0 direct 5 classes, merged", direct5_params, direct5_epochs, direct5_best["epoch"],
     [float(merged_iou.mean()), float(merged_iou[0]), float(merged_iou[1]), float(merged_iou[2])]),
    ("SegFormer-B0 3 classes, stage 1", segformer3_params, segformer3_epochs, segformer3_best["epoch"],
     [segformer3_best["miou"], segformer3_best["coral_iou"], segformer3_best["substrate_iou"], segformer3_best["other_iou"]]),
]
for label, params, epochs_trained, best_epoch, values in runs:
    rows.append({
        "model": label,
        "parameters": params,
        "epochs trained": epochs_trained,
        "best epoch": best_epoch,
        "mIoU": values[0],
        "coral IoU": values[1],
        "substrate IoU": values[2],
        "other IoU": values[3],
    })

print("Both models on the {} validation photos resized to {} by {}, scored against the 3-class masks of notebook 02".format(
    len(eval_data3), val_height, val_width))
print("The test split is not used. 5-class numbers: one pass of best.pt. 3-class numbers: {}".format(segformer3_source))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Observation**: